In [66]:
import pandas as pd

py_df = pd.read_csv('data/py_repos.csv', usecols=['rank', 'username/repo_name'])

class A:
    pass
method_names = sorted(dir(A))

py_df = pd.DataFrame({'rank': range(len(method_names)), 'username/repo_name': method_names})

all_safetensor_keys = set(f'{row["rank"]}.{row["username/repo_name"]}' for _, row in py_df.iterrows())
all_safetensor_keys

{'0.__class__',
 '1.__delattr__',
 '10.__gt__',
 '11.__hash__',
 '12.__init__',
 '13.__init_subclass__',
 '14.__le__',
 '15.__lt__',
 '16.__module__',
 '17.__ne__',
 '18.__new__',
 '19.__reduce__',
 '2.__dict__',
 '20.__reduce_ex__',
 '21.__repr__',
 '22.__setattr__',
 '23.__sizeof__',
 '24.__str__',
 '25.__subclasshook__',
 '26.__weakref__',
 '3.__dir__',
 '4.__doc__',
 '5.__eq__',
 '6.__format__',
 '7.__ge__',
 '8.__getattribute__',
 '9.__getstate__'}

In [67]:
from pathlib import Path

df_genesis = pd.read_csv('data/genesis.csv')
py_df = pd.DataFrame({'rank': df_genesis[df_genesis['chapter'] == 1]['verse'], 'username/repo_name': df_genesis[df_genesis['chapter'] == 1]['text']})

all_safetensor_keys = set(f'{row["rank"]}.{row["username/repo_name"]}' for _, row in py_df.iterrows())
all_safetensor_keys

{'1.In the beginning when God created the heavens and the earth,',
 '10.God called the dry land Earth, and the waters that were gathered together he called Seas. And God saw that it was good.',
 '11.Then God said, "Let the earth put forth vegetation: plants yielding seed, and fruit trees of every kind on earth that bear fruit with the seed in it." And it was so.',
 '12.The earth brought forth vegetation: plants yielding seed of every kind, and trees of every kind bearing fruit with the seed in it. And God saw that it was good.',
 '13.And there was evening and there was morning, the third day.',
 '14.And God said, "Let there be lights in the dome of the sky to separate the day from the night; and let them be for signs and for seasons and for days and years,',
 '15.and let them be lights in the dome of the sky to give light upon the earth." And it was so.',
 '16.God made the two great lights - the greater light to rule the day and the lesser light to rule the night - and the stars.',
 '1

In [68]:
from safetensors.torch import load_file
import polars as pl
from pathlib import Path
import torch


schema = [('expert', pl.Int8)]
for _, row in py_df.iterrows():
    repo = row["username/repo_name"]
    schema.append((f'{repo}.w1', pl.Float16))
    schema.append((f'{repo}.w3', pl.Float16))


layer_to_df: dict[int, pl.DataFrame] = {}

for layer in range(32):

    df = pl.DataFrame(schema=schema)
    layer_path = Path(f'output/genesis/layer={layer:02d}')

    for expert in range(8):
        repos_to_activations = {}
        expert_path = layer_path / f'expert={expert}'

        w1_path = expert_path / 'w=1.safetensors'
        w3_path = expert_path / 'w=3.safetensors'

        w1 = load_file(w1_path)
        w3 = load_file(w3_path)

        assert set(w1.keys()) == all_safetensor_keys, f"Repo missing in w=1 for expert {expert}"
        assert set(w3.keys()) == all_safetensor_keys, f"Repo missing in w=3 for expert {expert}"

        for _, row in py_df.iterrows():
            key = f'{row["rank"]}.{row["username/repo_name"]}'

            w1_activation = w1[key].cpu().to(torch.float16).numpy()
            assert w1_activation.shape == (14336,), f"Unexpected shape for {key} in w=1: {w1_activation.shape}"
            repos_to_activations[f'{row["username/repo_name"]}.w1'] = w1_activation


            w3_activation = w3[key].cpu().to(torch.float16).numpy()
            assert w3_activation.shape == (14336,), f"Unexpected shape for {key} in w=3: {w3_activation.shape}"
            repos_to_activations[f'{row["username/repo_name"]}.w3'] = w3_activation
        
        curr_expert_df = pl.DataFrame(repos_to_activations)
        curr_expert_df = curr_expert_df.with_columns(pl.lit(expert, dtype=pl.Int8).alias("expert"))
        curr_expert_df = curr_expert_df.select(pl.col(df.columns))


        df = pl.concat([df, curr_expert_df], how="vertical")
    
    layer_to_df[layer] = df

In [69]:
df = layer_to_df[16]

df_top_50_indices = df.drop('expert').select(
    pl.all().arg_sort(descending=True).head(10)
)

# df_top_50_indices = df.drop('expert').select(
#     pl.all().sort(descending=True).head(10)
# )

# df_top_50_indices
df_top_50_indices // 14336

"In the beginning when God created the heavens and the earth,.w1","In the beginning when God created the heavens and the earth,.w3","the earth was a formless void and darkness covered the face of the deep, while a wind from God swept over the face of the waters..w1","the earth was a formless void and darkness covered the face of the deep, while a wind from God swept over the face of the waters..w3","Then God said, ""Let there be light""; and there was light..w1","Then God said, ""Let there be light""; and there was light..w3",And God saw that the light was good; and God separated the light from the darkness..w1,And God saw that the light was good; and God separated the light from the darkness..w3,"God called the light Day, and the darkness he called Night. And there was evening and there was morning, the first day..w1","God called the light Day, and the darkness he called Night. And there was evening and there was morning, the first day..w3","And God said, ""Let there be a dome in the midst of the waters, and let it separate the waters from the waters."".w1","And God said, ""Let there be a dome in the midst of the waters, and let it separate the waters from the waters."".w3",So God made the dome and separated the waters that were under the dome from the waters that were above the dome. And it was so..w1,So God made the dome and separated the waters that were under the dome from the waters that were above the dome. And it was so..w3,"God called the dome Sky. And there was evening and there was morning, the second day..w1","God called the dome Sky. And there was evening and there was morning, the second day..w3","And God said, ""Let the waters under the sky be gathered together into one place, and let the dry land appear."" And it was so..w1","And God said, ""Let the waters under the sky be gathered together into one place, and let the dry land appear."" And it was so..w3","God called the dry land Earth, and the waters that were gathered together he called Seas. And God saw that it was good..w1","God called the dry land Earth, and the waters that were gathered together he called Seas. And God saw that it was good..w3","Then God said, ""Let the earth put forth vegetation: plants yielding seed, and fruit trees of every kind on earth that bear fruit with the seed in it."" And it was so..w1","Then God said, ""Let the earth put forth vegetation: plants yielding seed, and fruit trees of every kind on earth that bear fruit with the seed in it."" And it was so..w3","The earth brought forth vegetation: plants yielding seed of every kind, and trees of every kind bearing fruit with the seed in it. And God saw that it was good..w1","The earth brought forth vegetation: plants yielding seed of every kind, and trees of every kind bearing fruit with the seed in it. And God saw that it was good..w3","And there was evening and there was morning, the third day..w1","And there was evening and there was morning, the third day..w3","And God said, ""Let there be lights in the dome of the sky to separate the day from the night; and let them be for signs and for seasons and for days and years,.w1","And God said, ""Let there be lights in the dome of the sky to separate the day from the night; and let them be for signs and for seasons and for days and years,.w3","and let them be lights in the dome of the sky to give light upon the earth."" And it was so..w1","and let them be lights in the dome of the sky to give light upon the earth."" And it was so..w3",God made the two great lights - the greater light to rule the day and the lesser light to rule the night - and the stars..w1,God made the two great lights - the greater light to rule the day and the lesser light to rule the night - and the stars..w3,"God set them in the dome of the sky to give light upon the earth,.w1","God set them in the dome of the sky to give light upon the earth,.w3","to rule over the day and over the night, and to separate the light from the darkness. And God saw that it was good..w

In [70]:
from pathlib import Path
import re

lines = Path('genesis.txt').read_text().splitlines()

chapter, verse, text = [], [], []
for line in lines:
    # [chapter:verse] text
    match = re.match(r'\[(\d+):(\d+)\] (.+)', line)
    if not match:
        raise ValueError(f"Line does not match expected format: {line}")
    chapter.append(int(match.group(1)))
    verse.append(int(match.group(2)))
    text.append(match.group(3))

df_genesis = pd.DataFrame({'chapter': chapter, 'verse': verse, 'text': text})
df_genesis.to_csv('genesis.csv', index=False)

In [71]:
pd.read_csv('data/genesis.csv')

,chapter,verse,text
0,1,1,In the beginning when God created the heavens ...
1,1,2,the earth was a formless void and darkness cov...
2,1,3,"Then God said, ""Let there be light""; and there..."
3,1,4,And God saw that the light was good; and God s...
4,1,5,"God called the light Day, and the darkness he ..."
...,...,...,...
1528,50,22,"So Joseph remained in Egypt, he and his father..."
1529,50,23,Joseph saw Ephraim's children of the third gen...
1530,50,24,"Then Joseph said to his brothers, ""I am about ..."
1531,50,25,"So Joseph made the Israelites swear, saying, ""..."


In [82]:
# join to chapter and text
df_genesis = pd.read_csv('data/genesis.csv')
df_genesis_by_chapter = df_genesis.groupby('chapter').apply(lambda g: g.sort_values('verse')['text'].tolist())
df_genesis_by_chapter = df_genesis_by_chapter.apply(lambda verses: '\n'.join(verses))
df_genesis_by_chapter.to_frame(name='text').reset_index()


/tmp/ipykernel_119397/1620948220.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_genesis_by_chapter = df_genesis.groupby('chapter').apply(lambda g: g.sort_values('verse')['text'].tolist())


,chapter,text
0,1,In the beginning when God created the heavens ...
1,2,"Thus the heavens and the earth were finished, ..."
2,3,Now the serpent was more crafty than any other...
3,4,"Now the man knew his wife Eve, and she conceiv..."
4,5,This is the list of the descendants of Adam. W...
5,6,When people began to multiply on the face of t...
6,7,"Then the LORD said to Noah, ""Go into the ark, ..."
7,8,But God remembered Noah and all the wild anima...
8,9,"God blessed Noah and his sons, and said to the..."
9,10,"These are the descendants of Noah's sons, Shem..."
